# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"Identifier: {metadata.identifier}\n")
print("Fields: name, description, keywords, version, license, and so forth are available via the 'metadata' attribute.")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List the record sets and fields in the dataset
record_sets = list(metadata.recordSets)
print(f'Record sets found: {len(record_sets)}')
for i, rs in enumerate(record_sets):
    print(f"\nRecord Set {i+1}:")
    print(f"  @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', rs['@id'])}")
    print("  Fields:")
    for f in rs['fields']:
        print(f"    - @id: {f['@id']}, name: {f.get('name', f['@id'])}, type: {f.get('dataType','')}")

# Save one record set id and a couple of useful field ids for later
if len(record_sets) == 0:
    raise RuntimeError('No record sets found in the Croissant schema!')
record_set_id = record_sets[0]['@id']

# List field IDs
field_ids = [f['@id'] for f in record_sets[0]['fields']]
print(f"\nUsing Record Set: {record_set_id}")
print(f"Fields (@id): {field_ids}")

## 3. Data Extraction
Load data from the record set into a DataFrame for analysis. Use the record set and field `@id`s found above.

In [ ]:
# Extract data from each available record set identified above
all_record_set_ids = [rs['@id'] for rs in metadata.recordSets]
print(f'Record sets to extract: {all_record_set_ids}')

dataframes = dict()
for recset_id in all_record_set_ids:
    records = list(dataset.records(record_set=recset_id))
    dataframes[recset_id] = pd.DataFrame(records)

# Display columns and head for the first record set
print(f'Columns in record set {record_set_id}:')
print(dataframes[record_set_id].columns.tolist())
dataframes[record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering, normalization, and grouping. Demonstrate removing outliers and transformations for a numeric field.

In [ ]:
# Choose a likely numeric field (example: 'http://mlcommons.org/croissant/field/molecular_weight')
import numpy as np

df = dataframes[record_set_id]
# Try to auto-detect a numeric field
numeric_candidates = [col for col in df.columns if any(x in col.lower() for x in ['weight', 'abundance', 'count', 'mw', 'peptide', 'coverage', 'score'])]
print(f'Numeric candidate fields: {numeric_candidates}')
if numeric_candidates:
    numeric_field = numeric_candidates[0]
else:
    raise ValueError('No numeric field found!')

# For demo, set a threshold at the 10th percentile
threshold = df[numeric_field].quantile(0.1) if np.issubdtype(df[numeric_field].dtype, np.number) else 10
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df[[numeric_field]].head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized" ]].head())

# Try to find an interesting group field
group_field_candidates = [col for col in df.columns if col not in [numeric_field] and (df[col].dtype=='object' or df[col].dtype.name == 'category')]
print(f'Group field candidates: {group_field_candidates}')
if group_field_candidates:
    group_field = group_field_candidates[0]
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"Grouped data by {group_field} (showing mean {numeric_field} per group):")
    print(grouped_df.head())
else:
    print('No categorical column found for grouping!')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Simple visualizations: histogram and boxplot for the numeric field
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 5))
plt.subplot(1,2,1)
sns.histplot(filtered_df[numeric_field].dropna(), kde=True, bins=30)
plt.title(f'Histogram of {numeric_field}')
plt.xlabel(numeric_field)

plt.subplot(1,2,2)
sns.boxplot(x=filtered_df[numeric_field])
plt.title(f'Boxplot of {numeric_field}')
plt.xlabel(numeric_field)

plt.tight_layout()
plt.show()

# If a group_field is available, show violin plot
if 'group_field' in locals():
    plt.figure(figsize=(10,5))
    sns.violinplot(x=filtered_df[group_field], y=filtered_df[numeric_field])
    plt.title(f'{numeric_field} by {group_field}')
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR² mass spectrometry dataset using its Croissant schema, explored available record sets and fields, extracted tabular data, performed filtering, normalization, grouped by key attributes, and visualized distribution patterns. This process highlights key attributes of the dataset for further analysis or modeling.